In [1]:
# Imports, env vars, logging/warning suppression

%env MUJOCO_GL=egl
import datetime
import functools
import os
import time
import warnings

import jax
import jax.numpy as jp
import mediapy as media
import mujoco
import numpy as np
import warp as wp
from absl import logging
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from etils import epath
from mujoco_playground import registry, wrapper

import twmr  # noqa: F401 — registers TWMRLegFlat

wp.config.quiet = True
xla_flags = os.environ.get("XLA_FLAGS", "")
xla_flags += " --xla_gpu_triton_gemm_any=True"
os.environ["XLA_FLAGS"] = xla_flags
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
logging.set_verbosity(logging.WARNING)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
media.set_ffmpeg("/bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.pixi/envs/default/bin/ffmpeg")

print("jax:", jax.__version__, " |  device:", jax.default_backend())

env: MUJOCO_GL=egl
jax: 0.8.2  |  device: gpu


In [2]:
# ── Training config ───────────────────────────────────────────────────────────
ENV_NAME             = "TWMRLegFlat" 
SEED                 = 1
NUM_TIMESTEPS        = 100_000_000
EPISODE_LENGTH       = 250        # 5 s at ctrl_dt = 0.02 s/step
NUM_ENVS             = 1024
NUM_EVAL_ENVS        = 128
NUM_EVALS            = 20         # evals every 5 % → hits 5, 10, …, 100 %
UNROLL_LENGTH        = 10
BATCH_SIZE           = 256
NUM_MINIBATCHES      = 8
NUM_UPDATES_PER_BATCH = 8
LEARNING_RATE        = 5e-4
ENTROPY_COST         = 5e-3
DISCOUNTING          = 0.97
REWARD_SCALING       = 1.0
CLIPPING_EPSILON     = 0.3
MAX_GRAD_NORM        = 1.0
POLICY_HIDDEN        = (64, 64, 64)
VALUE_HIDDEN         = (64, 64, 64)
LOGDIR               = "logs"
# Render a video at these training percentages
TARGET_PCTS          = {5, 20, 50, 100}
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
# ── Logging directory ─────────────────────────────────────────────────────────
exp_name = f"{ENV_NAME}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
logdir   = epath.Path(LOGDIR).resolve() / exp_name
logdir.mkdir(parents=True, exist_ok=True)
ckpt_dir = logdir / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
print(f"Logs: {logdir}")

# ── Environments ──────────────────────────────────────────────────────────────
# The Warp backend uses nconmax as the TOTAL contact buffer across all parallel
# environments, not per-env. Scale it so training/eval batches don't overflow.
# (Error messages showed naconmax up to ~16 000 needed for 1024 envs → use 20×.)
NCONMAX_TRAIN = 20 * NUM_ENVS       # ~20 contacts/env × 1024 envs = 20 480
NCONMAX_EVAL  = 20 * NUM_EVAL_ENVS  # ~20 contacts/env × 128  envs =  2 560

env      = registry.load(ENV_NAME, config_overrides={"nconmax": NCONMAX_TRAIN})
eval_env = registry.load(ENV_NAME, config_overrides={"nconmax": NCONMAX_EVAL})
print(f"action_size={env.action_size}  obs_size={env.observation_size}")

# Separate env instance used only for checkpoint video rendering (single-env, default nconmax)
_render_env     = registry.load(ENV_NAME)
_render_wrapped = wrapper.wrap_for_brax_training(
    _render_env, episode_length=EPISODE_LENGTH, action_repeat=1
)


# ── Rollout rendering helper ──────────────────────────────────────────────────
def render_rollout(make_policy, params, outpath):
    """Run one deterministic episode and save as mp4."""
    inference_fn = make_policy(params, deterministic=True)
    jit_infer    = jax.jit(inference_fn)

    rng_batch = jax.random.split(jax.random.PRNGKey(SEED + 100), 1)
    state     = jax.jit(_render_wrapped.reset)(rng_batch)

    def step_fn(carry, _):
        s, rng = carry
        rng, k = jax.random.split(rng)
        ks     = jax.random.split(k, 1)
        act    = jax.vmap(jit_infer)(s.obs, ks)[0]
        s      = _render_wrapped.step(s, act)
        return (s, rng), s.data

    _, traj = jax.lax.scan(
        step_fn, (state, jax.random.PRNGKey(SEED)), None, length=EPISODE_LENGTH
    )
    traj.qpos.block_until_ready()

    qpos_np = np.array(traj.qpos[:, 0])  # (T, nq) for video 0
    qvel_np = np.array(traj.qvel[:, 0])  # (T, nv)

    mj_model   = _render_env.mj_model
    renderer   = mujoco.Renderer(mj_model, height=480, width=640)
    cam        = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(mj_model, cam)
    cam.azimuth, cam.elevation, cam.distance = 135.0, -20.0, 1.5
    chassis_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "chassis")
    d = mujoco.MjData(mj_model)

    frames = []
    for i in range(qpos_np.shape[0]):
        d.qpos[:] = qpos_np[i]
        d.qvel[:] = qvel_np[i]
        mujoco.mj_forward(mj_model, d)
        cam.lookat[:] = d.xpos[chassis_id]
        renderer.update_scene(d, camera=cam)
        frames.append(renderer.render())
    renderer.close()

    media.write_video(str(outpath), frames, fps=1.0 / _render_env.dt)
    media.show_video(frames, fps=1.0 / _render_env.dt)
    print(f"  Saved: {outpath}")
    return frames


Logs: /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/sandbox/logs/TWMRLegFlat-20260331-145129
action_size=8  obs_size=45


In [4]:
times           = [time.monotonic()]
saved_pcts      = set()
_sorted_targets = sorted(TARGET_PCTS)


def progress(num_steps, metrics):
    times.append(time.monotonic())
    reward = metrics.get("eval/episode_reward", float("nan"))
    pct    = num_steps / NUM_TIMESTEPS * 100
    print(f"[{pct:5.1f}%] step={num_steps:>9,}  reward={reward:.4f}")


def policy_params_fn(current_step, make_policy, params):
    # Trigger at the first eval that crosses each target threshold.
    # (Brax eval steps rarely land exactly on round percentages.)
    pct = current_step / NUM_TIMESTEPS * 100
    for target in _sorted_targets:
        if pct >= target and target not in saved_pcts:
            saved_pcts.add(target)
            print(f"→ Rendering checkpoint at {pct:.1f}% (target ≥{target}%)…")
            outpath = logdir / f"rollout_{target:03d}pct.mp4"
            render_rollout(make_policy, params, outpath)


train_fn = functools.partial(
    ppo.train,
    num_timesteps         = NUM_TIMESTEPS,
    num_evals             = NUM_EVALS,
    episode_length        = EPISODE_LENGTH,
    num_envs              = NUM_ENVS,
    unroll_length         = UNROLL_LENGTH,
    batch_size            = BATCH_SIZE,
    num_minibatches       = NUM_MINIBATCHES,
    num_updates_per_batch = NUM_UPDATES_PER_BATCH,
    learning_rate         = LEARNING_RATE,
    entropy_cost          = ENTROPY_COST,
    discounting           = DISCOUNTING,
    reward_scaling        = REWARD_SCALING,
    clipping_epsilon      = CLIPPING_EPSILON,
    max_grad_norm         = MAX_GRAD_NORM,
    normalize_observations= True,
    action_repeat         = 1,
    network_factory       = functools.partial(
        ppo_networks.make_ppo_networks,
        policy_hidden_layer_sizes = POLICY_HIDDEN,
        value_hidden_layer_sizes  = VALUE_HIDDEN,
    ),
    seed                  = SEED,
    save_checkpoint_path  = ckpt_dir,
    wrap_env_fn           = wrapper.wrap_for_brax_training,
    num_eval_envs         = NUM_EVAL_ENVS,
)

make_inference_fn, params, _ = train_fn(
    environment      = env,
    eval_env         = eval_env,
    progress_fn      = progress,
    policy_params_fn = policy_params_fn,
)

print(f"\nJIT compile time : {times[1] - times[0]:.1f}s")
print(f"Total train time : {times[-1] - times[1]:.1f}s")

[  0.0%] step=        0  reward=-8.6692
→ Rendering checkpoint at 5.3% (target ≥5%)…


This browser does not support the video tag.


ERROR:asyncio:Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib64/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x7f1f643e2b80> is already entered


  Saved: /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/sandbox/logs/TWMRLegFlat-20260331-145129/rollout_005pct.mp4
[  5.3%] step=5,263,360  reward=46.6961


ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-42' coro=<_async_in_context.<locals>.run_in_context() running at /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.venv/lib64/python3.11/site-packages/ipykernel/utils.py:60> wait_for=<Task pending name='Task-2' coro=<Kernel.shell_main() running at /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.venv/lib64/python3.11/site-packages/ipykernel/kernelbase.py:590> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.venv/lib64/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-2' coro=<Kernel.shell_main() running at /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/.venv/lib64/python3.11/site-packages/ipykernel/kernelbase.py:590> cb=[Task.task_wakeup()]>


...

  Saved: /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/sandbox/logs/TWMRLegFlat-20260331-145129/rollout_020pct.mp4
[ 21.1%] step=21,053,440  reward=50.7139
[ 26.3%] step=26,316,800  reward=50.5793
[ 31.6%] step=31,580,160  reward=52.6825
[ 36.8%] step=36,843,520  reward=52.3034
[ 42.1%] step=42,106,880  reward=51.8589
[ 47.4%] step=47,370,240  reward=49.8219
→ Rendering checkpoint at 52.6% (target ≥50%)…


This browser does not support the video tag.


  Saved: /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/sandbox/logs/TWMRLegFlat-20260331-145129/rollout_050pct.mp4
[ 52.6%] step=52,633,600  reward=53.5218
[ 57.9%] step=57,896,960  reward=54.7155
[ 63.2%] step=63,160,320  reward=52.8033
[ 68.4%] step=68,423,680  reward=50.7876
[ 73.7%] step=73,687,040  reward=51.7354
[ 79.0%] step=78,950,400  reward=47.7173
[ 84.2%] step=84,213,760  reward=48.0262
[ 89.5%] step=89,477,120  reward=47.4815
[ 94.7%] step=94,740,480  reward=46.3152
→ Rendering checkpoint at 100.0% (target ≥100%)…


This browser does not support the video tag.


  Saved: /bigdata/rhome/jbuh2025/Transformable-Leg-Wheel-Robot/sandbox/logs/TWMRLegFlat-20260331-145129/rollout_100pct.mp4
[100.0%] step=100,003,840  reward=46.1140

JIT compile time : 12.5s
Total train time : 803.3s


In [ ]:
print("Rendering final rollout…")
frames = render_rollout(make_inference_fn, params, logdir / "rollout_final.mp4")
media.show_video(frames, fps=1.0 / _render_env.dt)